# 01 — Synthetic Data Generation & EDA

## UPI Propensity Score API

**Objective:** Generate a realistic synthetic UPI propensity dataset, explore feature distributions, and prepare the training set.

### Why Synthetic Data?

Production propensity models train on real customer data from a CDP (Customer Data Platform). For this portfolio project, we generate synthetic data that preserves the statistical properties of real UPI behavioral data:
- **~5% positive rate** — realistic for dormant user reactivation campaigns
- **Correlated features** — users with high app engagement also tend to have recent transactions
- **Class imbalance** — requires scale_pos_weight and Precision@K evaluation
- **Realistic nulls** — balance_trend (15% null for new accounts), notification_ctr (5% null for disabled notifications)

The model architecture, evaluation framework, and API design are production-ready — only the data source changes in a real deployment.

---

### Feature Categories (30 features)

| Category | Count | Key Features |
|----------|-------|--------------|
| User Demographics | 4 | months_on_book, age_band, affluence_segment, num_products_held |
| UPI Transaction History | 6 | txn_count_30d, txn_count_90d, avg_txn_amount, days_since_last_txn, ... |
| Value Share | 2 | value_share_pct, primary_app_flag |
| Balance / Financial | 3 | avg_monthly_balance, balance_trend, low_balance_months_6m |
| App Engagement | 5 | app_opens_30d, days_since_last_app_open, notification_ctr, ... |
| Rewards | 4 | cashback_received_90d, cashback_redeemed_pct, milestones_completed, reward_tier |
| Fraud / Risk | 3 | fraud_flag_l1, fraud_flag_l2, chargeback_count_12m |
| Digital Behavior | 3 | has_upi_autopay, peer_txn_ratio, bill_pay_active |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')

DATA_PROCESSED = Path('../data/processed')
MODELS_DIR = Path('../models')

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Synthetic Data Generator

Uses a **latent propensity score** to create realistic feature correlations:
1. Each user gets a latent score ~ Uniform(0, 1)
2. Features are generated as f(latent) + noise — strong predictors get more latent signal
3. Target `will_transact` is derived via a logistic function on latent + noise, calibrated for ~5% positive rate

In [ ]:
def generate_upi_dataset(n=200_000, seed=42):
    """Generate synthetic UPI propensity dataset with realistic correlations."""
    rng = np.random.RandomState(seed)
    
    # Latent propensity score — drives all correlated features
    latent = rng.uniform(0, 1, size=n)
    
    # === User Demographics (weak signal) ===
    months_on_book = np.clip(rng.exponential(scale=30, size=n) + 1, 1, 120).astype(int)
    age_band = rng.choice([1, 2, 3, 4, 5], size=n, p=[0.15, 0.35, 0.25, 0.15, 0.10])
    affluence_raw = latent * 0.3 + rng.uniform(0, 1, size=n) * 0.7
    affluence_segment = np.digitize(affluence_raw, bins=[0, 0.60, 0.85, 0.95]) 
    num_products_held = np.clip(rng.poisson(lam=1.5 + latent * 1.5, size=n) + 1, 1, 6)
    
    # === UPI Transaction History (strong signal) ===
    txn_count_30d = rng.poisson(lam=np.maximum(0.5, latent * 30), size=n)
    dormant_mask = (latent < 0.3) & (rng.random(n) < 0.6)
    txn_count_30d[dormant_mask] = 0
    
    txn_count_90d = txn_count_30d * 3 + rng.poisson(lam=5, size=n)
    txn_count_90d = np.maximum(txn_count_90d, txn_count_30d)
    
    avg_txn_amount = np.exp(5.5 + latent * 1.5 + rng.normal(0, 1.2, size=n))
    avg_txn_amount = np.clip(avg_txn_amount, 50, 50000)
    
    total_txn_value_30d = txn_count_30d * avg_txn_amount * (0.6 + 0.8 * rng.random(n))
    total_txn_value_30d = np.where(txn_count_30d == 0, 0, total_txn_value_30d)
    
    days_since_last_txn = np.clip(
        rng.exponential(scale=5 + (1 - latent) * 150, size=n), 0, 365
    ).astype(int)
    
    txn_frequency_trend = np.clip(
        latent * 0.8 - 0.3 + rng.normal(0, 0.3, size=n), -1, 1
    )
    
    # === Value Share (strong signal) ===
    value_share_pct = np.clip(
        stats.beta.rvs(2 + latent * 3, 5, random_state=rng, size=n), 0, 1
    )
    primary_app_flag = (value_share_pct > 0.5).astype(int)
    
    # === Balance / Financial (moderate signal) ===
    avg_monthly_balance = np.exp(8.5 + latent * 2 + rng.normal(0, 1.5, size=n))
    avg_monthly_balance = np.clip(avg_monthly_balance, 500, 500000)
    
    balance_trend = np.clip(latent * 0.4 - 0.1 + rng.normal(0, 0.4, size=n), -1, 1)
    balance_trend_null = (months_on_book < 6) | (rng.random(n) < 0.10)
    balance_trend = np.where(balance_trend_null, np.nan, balance_trend)
    
    low_balance_months_6m = np.clip(rng.poisson(lam=2 - latent * 1.5, size=n), 0, 6)
    
    # === App Engagement (strong signal) ===
    app_opens_30d = rng.poisson(lam=np.maximum(1, latent * 25), size=n)
    app_opens_30d[latent < 0.2] = np.where(
        rng.random(np.sum(latent < 0.2)) < 0.5, 0, app_opens_30d[latent < 0.2]
    )
    
    app_opens_trend = np.clip(latent * 0.6 - 0.2 + rng.normal(0, 0.3, size=n), -1, 1)
    
    days_since_last_app_open = np.clip(
        rng.exponential(scale=3 + (1 - latent) * 80, size=n), 0, 180
    ).astype(int)
    
    session_duration_avg = np.clip(
        np.exp(3.5 + latent * 0.5 + rng.normal(0, 0.6, size=n)), 10, 600
    )
    
    notification_ctr = np.clip(latent * 0.2 + rng.normal(0, 0.08, size=n), 0, 0.5)
    notification_ctr = np.where(rng.random(n) < 0.05, np.nan, notification_ctr)
    
    # === Rewards (moderate signal) ===
    cashback_received_90d = np.where(
        txn_count_90d == 0, 0,
        np.clip(rng.exponential(scale=50 + latent * 200, size=n), 0, 2000)
    )
    cashback_received_90d = np.where(
        (latent < 0.4) & (rng.random(n) < 0.5), 0, cashback_received_90d
    )
    
    cashback_redeemed_pct = np.where(
        cashback_received_90d == 0, 0,
        np.clip(latent * 0.6 + rng.normal(0.3, 0.2, size=n), 0, 1)
    )
    
    milestones_completed = np.clip(rng.poisson(lam=0.5 + latent * 3, size=n), 0, 10)
    
    reward_tier_raw = latent * 0.2 + rng.uniform(0, 1, size=n) * 0.8
    reward_tier = np.digitize(reward_tier_raw, bins=[0, 0.50, 0.80, 0.95]) 
    
    # === Fraud / Risk (strong negative) ===
    fraud_flag_l1 = (rng.random(n) < (0.005 + (1 - latent) * 0.03)).astype(int)
    fraud_flag_l2 = (fraud_flag_l1 & (rng.random(n) < 0.25)).astype(int)
    
    chargeback_count_12m = np.where(
        rng.random(n) < 0.90, 0,
        rng.poisson(lam=0.5 + (1 - latent) * 1.5, size=n)
    )
    chargeback_count_12m = np.clip(chargeback_count_12m, 0, 5)
    
    # === Digital Behavior (moderate signal) ===
    has_upi_autopay = (rng.random(n) < (0.1 + latent * 0.25)).astype(int)
    peer_txn_ratio = np.clip(
        stats.beta.rvs(2, 3, random_state=rng, size=n) + latent * 0.1, 0, 1
    )
    bill_pay_active = (rng.random(n) < (0.2 + latent * 0.3)).astype(int)
    
    # === Target: will_transact (~5% positive rate) ===
    # Calibrated intercept: -9.0 + 8.0*mean(latent) ≈ -5.0 → ~5% after noise
    logit = -9.0 + 8.0 * latent + rng.normal(0, 1.0, size=n)
    logit -= fraud_flag_l1 * 2.0 + fraud_flag_l2 * 3.0
    prob = 1 / (1 + np.exp(-logit))
    will_transact = (rng.random(n) < prob).astype(int)
    
    df = pd.DataFrame({
        'user_id': [f'U{i:06d}' for i in range(n)],
        'months_on_book': months_on_book,
        'age_band': age_band,
        'affluence_segment': affluence_segment,
        'num_products_held': num_products_held,
        'txn_count_30d': txn_count_30d,
        'txn_count_90d': txn_count_90d,
        'avg_txn_amount': np.round(avg_txn_amount, 2),
        'total_txn_value_30d': np.round(total_txn_value_30d, 2),
        'days_since_last_txn': days_since_last_txn,
        'txn_frequency_trend': np.round(txn_frequency_trend, 4),
        'value_share_pct': np.round(value_share_pct, 4),
        'primary_app_flag': primary_app_flag,
        'avg_monthly_balance': np.round(avg_monthly_balance, 2),
        'balance_trend': np.round(balance_trend, 4),
        'low_balance_months_6m': low_balance_months_6m,
        'app_opens_30d': app_opens_30d,
        'app_opens_trend': np.round(app_opens_trend, 4),
        'days_since_last_app_open': days_since_last_app_open,
        'session_duration_avg': np.round(session_duration_avg, 2),
        'notification_ctr': np.round(notification_ctr, 4),
        'cashback_received_90d': np.round(cashback_received_90d, 2),
        'cashback_redeemed_pct': np.round(cashback_redeemed_pct, 4),
        'milestones_completed': milestones_completed,
        'reward_tier': reward_tier,
        'fraud_flag_l1': fraud_flag_l1,
        'fraud_flag_l2': fraud_flag_l2,
        'chargeback_count_12m': chargeback_count_12m,
        'has_upi_autopay': has_upi_autopay,
        'peer_txn_ratio': np.round(peer_txn_ratio, 4),
        'bill_pay_active': bill_pay_active,
        'will_transact': will_transact,
    })
    
    return df


df = generate_upi_dataset(n=200_000, seed=42)
print(f"Dataset shape: {df.shape}")
print(f"Positive rate: {df['will_transact'].mean():.2%}")
print(f"Positive count: {df['will_transact'].sum():,} / {len(df):,}")
df.head()

## 2. Target Distribution

In [ ]:
positive_rate = df['will_transact'].mean()
print(f"Positive rate (will transact if nudged): {positive_rate:.2%}")
print(f"\nClass distribution:")
print(df['will_transact'].value_counts())
print(f"\nImbalance ratio: {(1 - positive_rate) / positive_rate:.0f}:1")

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x='will_transact', data=df, ax=ax, palette=['#2ecc71', '#e74c3c'])
ax.set_title(f'Target Distribution (positive rate: {positive_rate:.2%})')
ax.set_xlabel('will_transact (0 = dormant, 1 = will transact)')
ax.set_ylabel('Count')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height()),
               ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.savefig('target_distribution.png', dpi=100)
plt.show()

## 3. Null Audit

In [ ]:
null_counts = df.isnull().sum()
null_cols = null_counts[null_counts > 0]

print("Columns with nulls:")
for col in null_cols.index:
    pct = null_cols[col] / len(df) * 100
    print(f"  {col}: {null_cols[col]:,} ({pct:.1f}%)")

print(f"\nAll other columns: 0 nulls")

## 4. Feature Distributions by Category

In [ ]:
# Transaction features — key predictors
txn_cols = ['txn_count_30d', 'txn_count_90d', 'avg_txn_amount', 'days_since_last_txn']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, txn_cols):
    for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
        subset = df[df['will_transact'] == label][col].dropna()
        ax.hist(subset, bins=50, alpha=0.5, color=color, label=f'will_transact={label}', density=True)
    ax.set_title(col)
    ax.legend()
plt.suptitle('Transaction Features by Target', fontsize=14)
plt.tight_layout()
plt.savefig('txn_features_dist.png', dpi=100)
plt.show()

In [ ]:
# Engagement features
eng_cols = ['app_opens_30d', 'days_since_last_app_open', 'value_share_pct', 'notification_ctr']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, eng_cols):
    for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
        subset = df[df['will_transact'] == label][col].dropna()
        ax.hist(subset, bins=50, alpha=0.5, color=color, label=f'will_transact={label}', density=True)
    ax.set_title(col)
    ax.legend()
plt.suptitle('Engagement Features by Target', fontsize=14)
plt.tight_layout()
plt.savefig('engagement_features_dist.png', dpi=100)
plt.show()

## 5. Correlation with Target

In [ ]:
# Correlation of all numeric features with will_transact
numeric_cols = df.select_dtypes(include=[np.number]).columns.drop(['will_transact'])
correlations = df[numeric_cols].corrwith(df['will_transact']).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#e74c3c' if v > 0 else '#3498db' for v in correlations]
correlations.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Feature Correlation with will_transact')
ax.set_xlabel('Pearson Correlation')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.savefig('feature_correlations.png', dpi=100)
plt.show()

print("\nTop 10 positive correlations:")
print(correlations.head(10).to_string())
print("\nTop 5 negative correlations:")
print(correlations.tail(5).to_string())

## 6. Feature Engineering & Selection

In [ ]:
# Log-transform skewed monetary features
df['avg_txn_amount_log'] = np.log1p(df['avg_txn_amount'])
df['avg_monthly_balance_log'] = np.log1p(df['avg_monthly_balance'])
df['total_txn_value_30d_log'] = np.log1p(df['total_txn_value_30d'])

# Fill nulls
df['balance_trend'] = df['balance_trend'].fillna(0.0)
df['notification_ctr'] = df['notification_ctr'].fillna(df['notification_ctr'].median())

# Final feature set (model-ready column names)
FEATURE_COLS = [
    # User demographics
    'months_on_book', 'age_band', 'affluence_segment', 'num_products_held',
    # UPI transaction history
    'txn_count_30d', 'txn_count_90d', 'avg_txn_amount_log',
    'total_txn_value_30d_log', 'days_since_last_txn', 'txn_frequency_trend',
    # Value share
    'value_share_pct', 'primary_app_flag',
    # Balance / financial
    'avg_monthly_balance_log', 'balance_trend', 'low_balance_months_6m',
    # App engagement
    'app_opens_30d', 'app_opens_trend', 'days_since_last_app_open',
    'session_duration_avg', 'notification_ctr',
    # Rewards
    'cashback_received_90d', 'cashback_redeemed_pct',
    'milestones_completed', 'reward_tier',
    # Fraud / risk
    'fraud_flag_l1', 'fraud_flag_l2', 'chargeback_count_12m',
    # Digital behavior
    'has_upi_autopay', 'peer_txn_ratio', 'bill_pay_active',
]

TARGET_COL = 'will_transact'

print(f"Total features: {len(FEATURE_COLS)}")
print(f"Null check: {df[FEATURE_COLS].isnull().sum().sum()} nulls remaining")

# Save feature names
with open(MODELS_DIR / 'feature_names.json', 'w') as f:
    json.dump(FEATURE_COLS, f, indent=2)
print(f"Saved feature names to {MODELS_DIR / 'feature_names.json'}")

## 7. Save Processed Data

In [ ]:
output_path = DATA_PROCESSED / 'features.parquet'
df[FEATURE_COLS + [TARGET_COL]].to_parquet(output_path, index=False)

print(f"Saved processed features to {output_path}")
print(f"Shape: {df[FEATURE_COLS + [TARGET_COL]].shape}")
print(f"\nFeature summary:")
print(df[FEATURE_COLS].describe().T[['mean', 'std', 'min', 'max']].round(2).to_string())